In [ ]:
Step 1 — Load and inspect the master dataset

In [6]:
# =========================================================
# 1. IMPORTS
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

In [9]:
# =========================================================
# 2. LOADED The CLEANED MASTER DATASET
# =========================================================

DATA_PATH = Path("multilingual_ticket_master_cleaned.csv")

print("Dataset exists:", DATA_PATH.exists())
print("Path:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

Dataset exists: True
Path: multilingual_ticket_master_cleaned.csv

Dataset loaded successfully.
Shape: (44278, 21)

Columns:
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'business_type', 'version', 'source_dataset', 'source_count', 'duplicate_record_count', 'label_conflict', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

Data types:
subject                    object
body                       object
answer                     object
type                       object
queue                      object
priority                   object
language                   object
business_type              object
version                   float64
source_dataset             object
source_count                int64
duplicate_record_count      int64
label_conflict               bool
tag_1                      object
tag_2                      object
tag_3                      object
tag_4                      object
tag_5                      object
t

/tmp/ipykernel_239/3687324812.py:10: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,subject,body,answer,type,queue,priority,language,business_type,version,source_dataset,source_count,duplicate_record_count,label_conflict,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Problema crítico del servidor requiere atención inmediata,Es necesaria una investigación inmediata sobre la interrupción en el servicio de gestión de AWS que está impactando funciones comerciales esenciales.,Estamos investigando urgentemente el problema con el servicio de gestión de AWS. Proporcionaremos actualizaciones sobre nuestro progreso.,Incident,Technical Support,high,es,IT Services,NaN,multi-lang3-4k,1,1,False,Urgent Issue,Service Disruption,Incident Report,Service Recovery,System Maintenance,NaN,NaN,NaN
1,Anfrage zur Verfügbarkeit des Dell XPS 13 9310,"Sehr geehrter Kundenservice, ich hoffe, diese E-Mail erreicht Sie wohl. Ich schreibe, um mich nach der Verfügbarkeit und den Lieferoptionen für da...","Sehr geehrter <name>, vielen Dank, dass Sie uns kontaktiert haben. Das Dell XPS 13 9310 Ultrabook ist derzeit verfügbar. Die geschätzte Lieferzeit...",Request,Customer Service,low,de,Tech Online Store,NaN,multi-lang3-4k,1,1,False,Sales Inquiry,Product Support,Customer Service,Order Issue,Returns and Exchanges,NaN,NaN,NaN
2,Erro na Autocompletação de Código do IntelliJ IDEA,"Prezado Suporte ao Cliente <name>, Estou escrevendo para trazer à sua atenção um problema recorrente com o recurso de autocompletação de código no...","Prezado <name>, Obrigado por entrar em contato e nos informar sobre o problema com o recurso de autocompletação de código no IntelliJ IDEA 2024.1....",Incident,Technical Support,high,pt,IT Services,NaN,multi-lang3-4k,1,1,False,Technical Support,Software Bug,Problem Resolution,Urgent Issue,IT Support,NaN,NaN,NaN
3,Urgent Assistance Required: AWS Service,"Dear IT Services Support Team, I am reaching out regarding a high-priority ticket related to our AWS Management Service. We are experiencing signi...","Dear <name>, Thank you for reaching out regarding your AWS Management Service issues. I understand the urgency and the impact on your operational ...",Request,IT Support,high,en,IT Services,NaN,multi-lang3-4k,1,1,False,IT Support,Urgent Issue,Service Notification,Cloud Services,Problem Resolution,Technical Guidance,Performance Tuning,NaN
4,Problème d'affichage de MacBook Air,"Cher équipe de support du magasin en ligne Tech, Je vous écris pour attirer votre attention sur un problème que j'ai rencontré avec mon MacBook Ai...","Cher <name>, Merci de nous avoir contactés concernant le problème avec l'affichage de votre MacBook Air M1. Nous sommes désolés d'apprendre le pro...",Incident,Product Support,low,fr,Tech Online Store,NaN,multi-lang3-4k,1,1,False,Technical Support,Product Support,Hardware Failure,Service Recovery,Routine Request,NaN,NaN,NaN


In [ ]:
The Below code will establish the basline

In [10]:
# Missing-value profile
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
    "unique_values": df.nunique()
})

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percent,unique_values
tag_8,41553,93.85,561
business_type,40278,90.97,9
tag_7,37404,84.48,820
tag_6,29806,67.32,1001
tag_5,16754,37.84,1038
version,15692,35.44,3
subject,4589,10.36,36968
tag_4,3515,7.94,927
tag_3,196,0.44,658
tag_2,59,0.13,405


In [ ]:
#inspect the main variable 

In [11]:
for col in ["queue", "type", "priority", "language"]:
    print(f"\n--- {col.upper()} ---")
    print(df[col].value_counts())


--- QUEUE ---
queue
Technical Support                  13095
Product Support                     8113
Customer Service                    6741
IT Support                          5237
Billing and Payments                4368
Returns and Exchanges               2204
Service Outages and Maintenance     1710
Sales and Pre-Sales                 1393
Human Resources                      802
General Inquiry                      615
Name: count, dtype: int64

--- TYPE ---
type
Incident    17758
Request     12714
Problem      9271
Change       4535
Name: count, dtype: int64

--- PRIORITY ---
priority
medium    17961
high      17355
low        8962
Name: count, dtype: int64

--- LANGUAGE ---
language
en    25138
de    17379
es      812
fr      476
pt      473
Name: count, dtype: int64


In [ ]:
Step 2 — Define the target -- for our routing module

In [12]:
TARGET = "queue"

print("Target:", TARGET)
print("\nTarget distribution:")
print(df[TARGET].value_counts())

print("\nTarget percentages:")
print(
    df[TARGET]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target: queue

Target distribution:
queue
Technical Support                  13095
Product Support                     8113
Customer Service                    6741
IT Support                          5237
Billing and Payments                4368
Returns and Exchanges               2204
Service Outages and Maintenance     1710
Sales and Pre-Sales                 1393
Human Resources                      802
General Inquiry                      615
Name: count, dtype: int64

Target percentages:
queue
Technical Support                  29.57
Product Support                    18.32
Customer Service                   15.22
IT Support                         11.83
Billing and Payments                9.86
Returns and Exchanges               4.98
Service Outages and Maintenance     3.86
Sales and Pre-Sales                 3.15
Human Resources                     1.81
General Inquiry                     1.39
Name: proportion, dtype: float64


In [ ]:
Step 3 — Create the core text feature -- This is the first real engineered feature.

In [13]:
# Preserve information about whether a subject existed
df["has_subject"] = df["subject"].notna().astype(int)

# Fill missing text safely
df["subject_clean"] = df["subject"].fillna("").astype(str).str.strip()
df["body_clean"] = df["body"].fillna("").astype(str).str.strip()

# Combine subject and body
df["combined_text"] = (
    df["subject_clean"] + " " + df["body_clean"]
).str.strip()

# Check result
display(
    df[
        ["subject", "body", "has_subject", "combined_text"]
    ].head()
)

,subject,body,has_subject,combined_text
0,Problema crítico del servidor requiere atención inmediata,Es necesaria una investigación inmediata sobre la interrupción en el servicio de gestión de AWS que está impactando funciones comerciales esenciales.,1,Problema crítico del servidor requiere atención inmediata Es necesaria una investigación inmediata sobre la interrupción en el servicio de gestión...
1,Anfrage zur Verfügbarkeit des Dell XPS 13 9310,"Sehr geehrter Kundenservice, ich hoffe, diese E-Mail erreicht Sie wohl. Ich schreibe, um mich nach der Verfügbarkeit und den Lieferoptionen für da...",1,"Anfrage zur Verfügbarkeit des Dell XPS 13 9310 Sehr geehrter Kundenservice, ich hoffe, diese E-Mail erreicht Sie wohl. Ich schreibe, um mich nach ..."
2,Erro na Autocompletação de Código do IntelliJ IDEA,"Prezado Suporte ao Cliente <name>, Estou escrevendo para trazer à sua atenção um problema recorrente com o recurso de autocompletação de código no...",1,"Erro na Autocompletação de Código do IntelliJ IDEA Prezado Suporte ao Cliente <name>, Estou escrevendo para trazer à sua atenção um problema recor..."
3,Urgent Assistance Required: AWS Service,"Dear IT Services Support Team, I am reaching out regarding a high-priority ticket related to our AWS Management Service. We are experiencing signi...",1,"Urgent Assistance Required: AWS Service Dear IT Services Support Team, I am reaching out regarding a high-priority ticket related to our AWS Manag..."
4,Problème d'affichage de MacBook Air,"Cher équipe de support du magasin en ligne Tech, Je vous écris pour attirer votre attention sur un problème que j'ai rencontré avec mon MacBook Ai...",1,"Problème d'affichage de MacBook Air Cher équipe de support du magasin en ligne Tech, Je vous écris pour attirer votre attention sur un problème qu..."


In [ ]:
Step 4 — Engineer structural text features - Driving simple but meaningful characterstics of the ticket. 
These give the model contextual information about ticket size and complexity without interpreting the actual language.

In [14]:
# Character counts
df["subject_char_count"] = df["subject_clean"].str.len()
df["body_char_count"] = df["body_clean"].str.len()
df["text_char_count"] = df["combined_text"].str.len()

# Word counts
df["subject_word_count"] = (
    df["subject_clean"]
    .str.split()
    .str.len()
)

df["body_word_count"] = (
    df["body_clean"]
    .str.split()
    .str.len()
)

df["text_word_count"] = (
    df["combined_text"]
    .str.split()
    .str.len()
)

# Preview engineered features
display(
    df[
        [
            "has_subject",
            "subject_word_count",
            "body_word_count",
            "text_word_count",
            "text_char_count"
        ]
    ].head()
)

,has_subject,subject_word_count,body_word_count,text_word_count,text_char_count
0,1,7,21,28,207
1,1,8,69,77,496
2,1,8,106,114,789
3,1,5,132,137,965
4,1,5,196,201,1352


In [ ]:
Step 5 — Normalize categorical features -- 

In [15]:
# Normalize language
df["language"] = (
    df["language"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Normalize priority
df["priority"] = (
    df["priority"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Normalize type
df["type"] = (
    df["type"]
    .astype(str)
    .str.strip()
)

# Verify
print("Languages:", sorted(df["language"].unique()))
print("Priorities:", sorted(df["priority"].unique()))
print("Types:", sorted(df["type"].unique()))

Languages: ['de', 'en', 'es', 'fr', 'pt']
Priorities: ['high', 'low', 'medium']
Types: ['Change', 'Incident', 'Problem', 'Request']


In [ ]:
Step 6 — Check potential leakage -- This portion of the code is for feature-leakage analysis. 

In [18]:
# Candidate features that could potentially leak routing information
candidate_features = [
    "type",
    "priority",
    "business_type",
    "version",
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4",
    "tag_5",
    "tag_6",
    "tag_7",
    "tag_8"
]

for col in candidate_features:
    if col in df.columns:
        print(f"\n{'='*60}")
        print(f"FEATURE: {col}")
        print(f"{'='*60}")

        print("Unique values:", df[col].nunique(dropna=True))
        print(
            "Missing %:",
            round(df[col].isna().mean() * 100, 2)
        )

        print("\nTop values:")
        print(df[col].value_counts(dropna=False).head(15))


FEATURE: type
Unique values: 4
Missing %: 0.0

Top values:
type
Incident    17758
Request     12714
Problem      9271
Change       4535
Name: count, dtype: int64

FEATURE: priority
Unique values: 3
Missing %: 0.0

Top values:
priority
medium    17961
high      17355
low        8962
Name: count, dtype: int64

FEATURE: business_type
Unique values: 9
Missing %: 90.97

Top values:
business_type
NaN                             40278
IT Services                      1716
Tech Online Store                1395
IT Consulting Firm                391
Software Development Company      311
Online Store                      171
IT Consulting Service               7
Pit Services                        5
_IT_Services_                       2
Adobe Photoshop 2024                2
Name: count, dtype: int64

FEATURE: version
Unique values: 3
Missing %: 35.44

Top values:
version
400.0    18598
NaN      15692
52.0      9119
51.0       869
Name: count, dtype: int64

FEATURE: tag_1
Unique values: 251
Missi

In [19]:
#Inspecting whether type & Priority appear tightly coupled with queue

# Queue vs Type
queue_type = pd.crosstab(
    df["queue"],
    df["type"],
    normalize="index"
).round(3)

display(queue_type)

# Queue vs Priority
queue_priority = pd.crosstab(
    df["queue"],
    df["priority"],
    normalize="index"
).round(3)

display(queue_priority)

type,Change,Incident,Problem,Request
queue,,,,
Billing and Payments,0.058,0.241,0.217,0.484
Customer Service,0.072,0.257,0.206,0.465
General Inquiry,0.281,0.320,0.159,0.239
Human Resources,0.071,0.515,0.110,0.304
IT Support,0.169,0.363,0.260,0.207
Product Support,0.116,0.432,0.223,0.229
Returns and Exchanges,0.127,0.383,0.206,0.284
Sales and Pre-Sales,0.176,0.219,0.132,0.473
Service Outages and Maintenance,0.245,0.547,0.060,0.147


priority,high,low,medium
queue,,,
Billing and Payments,0.298,0.214,0.488
Customer Service,0.188,0.328,0.483
General Inquiry,0.128,0.584,0.288
Human Resources,0.108,0.481,0.410
IT Support,0.481,0.098,0.421
Product Support,0.304,0.196,0.500
Returns and Exchanges,0.211,0.373,0.416
Sales and Pre-Sales,0.164,0.360,0.477
Service Outages and Maintenance,0.715,0.109,0.175


In [ ]:
Step 7 — Explicitly execlude obvious leakage/metadata

In [20]:
TARGET = "queue"

exclude_features = [
    "answer",                  # post-ticket response → leakage
    "business_type",           # extremely sparse
    "version",                 # high missingness initially
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4",
    "tag_5",
    "tag_6",
    "tag_7",
    "tag_8",
    "source_dataset",          # provenance metadata
    "source_count",            # dataset construction metadata
    "duplicate_record_count",  # dataset construction metadata
    "label_conflict"           # QC metadata
]

print("Excluded features:")
for feature in exclude_features:
    print("-", feature)

Excluded features:
- answer
- business_type
- version
- tag_1
- tag_2
- tag_3
- tag_4
- tag_5
- tag_6
- tag_7
- tag_8
- source_dataset
- source_count
- duplicate_record_count
- label_conflict


In [ ]:
Step 8 — Create the baseline feature set

In [24]:
# Assuming type & Priority are available at ticket creation 
TEXT_FEATURE = "combined_text"

CATEGORICAL_FEATURES = [
    "language",
    "priority",
    "type"
]

NUMERIC_FEATURES = [
    "has_subject",
    "subject_char_count",
    "body_char_count",
    "text_char_count",
    "subject_word_count",
    "body_word_count",
    "text_word_count"
]

TARGET = "queue"

model_columns = (
    [TEXT_FEATURE]
    + CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + [TARGET]
)

model_df = df[model_columns].copy()

print("Model dataset shape:", model_df.shape)
print("\nModel columns:")
print(model_df.columns.tolist())

print("\nMissing values:")
print(model_df.isnull().sum())

Model dataset shape: (44278, 12)

Model columns:
['combined_text', 'language', 'priority', 'type', 'has_subject', 'subject_char_count', 'body_char_count', 'text_char_count', 'subject_word_count', 'body_word_count', 'text_word_count', 'queue']

Missing values:
combined_text         0
language              0
priority              0
type                  0
has_subject           0
subject_char_count    0
body_char_count       0
text_char_count       0
subject_word_count    0
body_word_count       0
text_word_count       0
queue                 0
dtype: int64


In [ ]:
Step 9 — Split before TF-IDF -- This order matters, Do not fit TF-IDF on the entire dataset before splitting,
because vocabulary/IDF information from the test set would leak into training.

In [25]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=[TARGET])
y = model_df[TARGET]

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

Training shape: (35422, 11)
Testing shape: (8856, 11)

Training target distribution:
queue
Technical Support                  0.296
Product Support                    0.183
Customer Service                   0.152
IT Support                         0.118
Billing and Payments               0.099
Returns and Exchanges              0.050
Service Outages and Maintenance    0.039
Sales and Pre-Sales                0.031
Human Resources                    0.018
General Inquiry                    0.014
Name: proportion, dtype: float64


In [ ]:
Step 10 — TF-IDF + categorical + numeric pipeline -- Coverting our diffrent feature types appropriately

Step 10 block of code iis doing three things simultaneously -- 

combined_text
     ↓
TF-IDF
     ↓
30,000-dimensional sparse text representation

language / priority / type
     ↓
One-Hot Encoding

word counts / character counts / has_subject
     ↓
StandardScaler

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Text transformer
text_transformer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Combined preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            text_transformer,
            TEXT_FEATURE
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            CATEGORICAL_FEATURES
        ),
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES
        )
    ]
)

In [ ]:
Step 11 — Logistic Regression baseline

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="saga",
                random_state=42
            )
        )
    ]
)

print("Baseline pipeline created successfully.")

Baseline pipeline created successfully.


In [32]:
import time

print("Starting model training...")

start_time = time.time()

baseline_model.fit(
    X_train,
    y_train
)

elapsed_time = time.time() - start_time

print("\nTraining complete!")
print(f"Training time: {elapsed_time:.2f} seconds")



Starting model training...

Training complete!
Training time: 177.61 seconds


In [35]:
#Predict -- 
y_pred = baseline_model.predict(X_test)

print("Prediction complete!")
print("Number of predictions:", len(y_pred))

Prediction complete!
Number of predictions: 8856


In [36]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

Accuracy:    0.4344
Macro F1:    0.4079
Weighted F1: 0.4386

Classification Report:
                                 precision    recall  f1-score   support

           Billing and Payments       0.68      0.68      0.68       874
               Customer Service       0.39      0.37      0.38      1348
                General Inquiry       0.18      0.50      0.26       123
                Human Resources       0.23      0.59      0.34       160
                     IT Support       0.37      0.41      0.39      1047
                Product Support       0.42      0.28      0.33      1623
          Returns and Exchanges       0.33      0.45      0.38       441
            Sales and Pre-Sales       0.24      0.50      0.32       279
Service Outages and Maintenance       0.38      0.70      0.49       342
              Technical Support       0.58      0.44      0.50      2619

                       accuracy                           0.43      8856
                      macro avg       

In [ ]:
44,278 cleaned multilingual tickets
→ engineered structural features
→ leakage-controlled feature selection
→ subject + body text representation
→ TF-IDF unigrams/bigrams
→ categorical encoding
→ numerical scaling
→ stratified train/test split
→ 10-class Logistic Regression routing model

Clean Master Dataset → Feature Engineering → Feature Transformation → Train/Test Split → Baseline Model → Evaluation


In [ ]:
RAW TICKET
    │
    ├── subject
    ├── body
    ├── language
    ├── priority
    └── type
         │
         ▼
FEATURE ENGINEERING
         │
         ├── combined_text
         ├── has_subject
         ├── subject_word_count
         ├── body_word_count
         ├── text_word_count
         ├── subject_char_count
         ├── body_char_count
         └── text_char_count
         │
         ▼
MODEL REPRESENTATION
         │
         ├── Text → TF-IDF / multilingual embeddings
         ├── Language → encoding
         ├── Priority → encoding
         ├── Type → encoding
         └── Queue → target encoding
         │
         ▼
     TRAIN / VALIDATION / TEST